In [52]:
import pandas as pd
import numpy as np

df=pd.read_csv("D:/ecomai/data/raw/data.csv",
    encoding="latin1"
)

In [53]:
print("Shape", df.shape)

Shape (541909, 8)


In [54]:
print("Missing Values")
print(df.isnull().sum())

Missing Values
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [55]:
print("Duplicated data :")
print(df.duplicated().sum())

Duplicated data :
5268


In [56]:
print("Data Types:")
print(df.dtypes)

Data Types:
InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object


In [57]:
print("Description Basic Stats:")
print(df.describe())


Description Basic Stats:
            Quantity      UnitPrice     CustomerID
count  541909.000000  541909.000000  406829.000000
mean        9.552250       4.611114   15287.690570
std       218.081158      96.759853    1713.600303
min    -80995.000000  -11062.060000   12346.000000
25%         1.000000       1.250000   13953.000000
50%         3.000000       2.080000   15152.000000
75%        10.000000       4.130000   16791.000000
max     80995.000000   38970.000000   18287.000000


In [58]:
# 135,080 missing = no identity = useless for churn
# We DROP because we cannot guess a customer's ID
before=df.shape[0]
print(before)
df=df[df['CustomerID'].notna()]
after=df.shape[0]
print(f"Removed customers whose customerIDs were missing :{before-after}")

541909
Removed customers whose customerIDs were missing :135080


In [61]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df['InvoiceDate'].dtype)  # Should show datetime64
print(df['InvoiceDate'].min())  # Earliest transaction
print(df['InvoiceDate'].max())  # Latest transaction


datetime64[us]
2010-12-01 08:26:00
2011-12-09 12:50:00


In [63]:
df['CustomerID']=df['CustomerID'].astype(int)
print(df.dtypes)

InvoiceNo                 str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID              int64
Country                   str
dtype: object


In [65]:
#Save returned Items/Orders
returns_df = df[df['Quantity'] < 0].copy()
returns_df.to_csv('D:/ecomai/data/processed/returns_data.csv', index=False)
print(f"Returns saved: {returns_df.shape[0]} rows")


Returns saved: 8905 rows


In [66]:
print("UnitPrice stats BEFORE:")
print(df['UnitPrice'].describe())

df = df[df['UnitPrice'] > 0]    
print("\nUnitPrice stats AFTER:")
print(df['UnitPrice'].describe())
# Min should now be above 0

UnitPrice stats BEFORE:
count    406829.000000
mean          3.460471
std          69.315162
min           0.000000
25%           1.250000
50%           1.950000
75%           3.750000
max       38970.000000
Name: UnitPrice, dtype: float64

UnitPrice stats AFTER:
count    406789.000000
mean          3.460811
std          69.318561
min           0.001000
25%           1.250000
50%           1.950000
75%           3.750000
max       38970.000000
Name: UnitPrice, dtype: float64


In [67]:
# Remove Cancelled Invoices

cancelled = df[df['InvoiceNo'].str.startswith('C')].shape[0]
print(f"Cancelled invoices found: {cancelled}")

df = df[~df['InvoiceNo'].str.startswith('C')]
print(f"Remaining rows: {df.shape[0]}")

Cancelled invoices found: 8905
Remaining rows: 397884


In [69]:
# Check for extreme outliers in Quantity and UnitPrice
print("Quantity outliers:")
Q1 = df['Quantity'].quantile(0.25)
Q3 = df['Quantity'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 3 * IQR
print(f"Upper limit: {upper_limit}")
print(f"Rows above limit: {df[df['Quantity'] > upper_limit].shape[0]}")

# flagging them — not removed yet
df['Quantity_Outlier'] = df['Quantity'] > upper_limit

Quantity outliers:
Upper limit: 42.0
Rows above limit: 18527


In [70]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# Verify
print(df[['Quantity', 'UnitPrice', 'TotalPrice']].head())
print("\nTotalPrice stats:")
print(df['TotalPrice'].describe())
print(f"\nAny zero or negative TotalPrice: {df[df['TotalPrice'] <= 0].shape[0]}")

   Quantity  UnitPrice  TotalPrice
0         6       2.55       15.30
1         6       3.39       20.34
2         8       2.75       22.00
3         6       3.39       20.34
4         6       3.39       20.34

TotalPrice stats:
count    397884.000000
mean         22.397000
std         309.071041
min           0.001000
25%           4.680000
50%          11.800000
75%          19.800000
max      168469.600000
Name: TotalPrice, dtype: float64

Any zero or negative TotalPrice: 0


In [71]:
print("="*40)
print("FINAL DATA HEALTH CHECK")
print("="*40)
print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nDate range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"Unique customers: {df['CustomerID'].nunique()}")
print(f"Unique countries: {df['Country'].nunique()}")

FINAL DATA HEALTH CHECK
Total rows: 397884
Total columns: 10

Missing values:
InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID          0
Country             0
Quantity_Outlier    0
TotalPrice          0
dtype: int64

Data types:
InvoiceNo                      str
StockCode                      str
Description                    str
Quantity                     int64
InvoiceDate         datetime64[us]
UnitPrice                  float64
CustomerID                   int64
Country                        str
Quantity_Outlier              bool
TotalPrice                 float64
dtype: object

Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00
Unique customers: 4338
Unique countries: 37


In [73]:
df.to_csv('D:/ecomai/data/processed/cleaned.csv', index=False)

print("Clean data saved successfully!")

Clean data saved successfully!
